# Day 084 — Exercise 4: run_duo — The Pipeline

**What you'll build:** `run_duo` — a free function that chains the ResearcherAgent and WriterAgent into a pipeline, creating a `Handoff` at each transition.

**Why it matters:** `run_duo` is where the two agents actually collaborate. The researcher gathers findings; a Handoff carries them to the writer; the writer produces the document; a second Handoff records the output. The return dict gives the caller everything — the intermediate findings, the final document, and the audit trail.

In [ ]:
def _mock_multi(findings='Finding: AI agents collaborate.', document='Doc: Agents work together.'):
    """Branch on system message: 'research specialist' -> findings, else -> document."""
    def _fn(messages):
        system = messages[0]['content'] if messages else ''
        if 'research specialist' in system.lower():
            return findings
        return document
    return _fn
import json

# ── helpers reused from Day 79 ───────────────────────────────────────────────
def safe_parse_json(text):
    """Slice first '{' to last '}' and parse. Returns dict|None (Day 79)."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, dict) else None


def call_llm(messages, llm_fn=None):
    """Call the chat model, or the injected llm_fn(messages) -> str (Day 79)."""
    if llm_fn is not None:
        return llm_fn(messages)
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]

# ── researcher specialist ─────────────────────────────────────────────────────
def build_researcher_prompt(query):
    """Build a prompt for the researcher role: gather facts on a topic."""
    system = "\n".join([
        "You are a research specialist. Your job is to gather relevant facts and",
        "key information about the topic given to you.",
        "",
        "Return a structured list of the most important findings.",
        "Be factual, concise, and cover the main points.",
    ])
    return [{"role": "system", "content": system},
            {"role": "user", "content": "Research topic: " + str(query)}]


class ResearcherAgent:
    """A specialist that researches a topic and returns structured findings.

    Each call to research() returns a string of findings and records the
    exchange in history. The agent has one job: gather facts. It passes its
    output to the next agent via a Handoff — it does not write, review, or
    plan.

    Example::

        researcher = ResearcherAgent(llm_fn=my_llm_fn)
        findings = researcher.research("topological sort algorithms")
    """

    def __init__(self, llm_fn=None):
        self._llm_fn = llm_fn
        self._history = []

    def research(self, query):
        """Research a query and return findings as a string."""
        messages = build_researcher_prompt(query)
        findings = call_llm(messages, llm_fn=self._llm_fn)
        self._history.append({"query": query, "findings": findings})
        return findings

    def history(self):
        """Return a copy of the research history."""
        return list(self._history)

    def clear_history(self):
        """Clear the history in place."""
        self._history.clear()

# ── writer specialist ─────────────────────────────────────────────────────────
def build_writer_prompt(findings, style="concise", instructions=None):
    """Build a prompt for the writer role: turn findings into a document."""
    system_parts = [
        "You are a writing specialist. Turn the provided findings into",
        "a polished, well-structured document.",
        "Style: " + str(style) + ".",
    ]
    if instructions:
        system_parts.append("Additional instructions: " + str(instructions))
    system = "\n".join(system_parts)
    user = "Findings:\n" + str(findings)
    return [{"role": "system", "content": system},
            {"role": "user", "content": user}]


class WriterAgent:
    """A specialist that turns research findings into a polished document.

    The writer has one job: take findings (a string from the ResearcherAgent
    or any other source) and produce a well-structured document. The style
    controls the tone (e.g. 'concise', 'detailed', 'formal').

    Example::

        writer = WriterAgent(llm_fn=my_llm_fn, style="concise")
        document = writer.write(findings)
    """

    def __init__(self, llm_fn=None, style="concise"):
        self._llm_fn = llm_fn
        self.style = style
        self._history = []

    def write(self, findings, instructions=None):
        """Write a document from findings. Returns the document string."""
        messages = build_writer_prompt(findings, style=self.style,
                                       instructions=instructions)
        document = call_llm(messages, llm_fn=self._llm_fn)
        self._history.append({"findings": findings, "document": document})
        return document

    def history(self):
        """Return a copy of the writing history."""
        return list(self._history)

    def clear_history(self):
        """Clear the history in place."""
        self._history.clear()

# ── handoffs: explicit data passing between agents ────────────────────────────
from dataclasses import dataclass, field


@dataclass
class Handoff:
    """An explicit record of data passed from one agent to the next.

    Attributes:
        from_agent: name of the sending agent ('researcher', 'writer', ...).
        to_agent:   name of the receiving agent.
        content:    the data being passed (findings string, document, ...).
        metadata:   optional dict for any extra context (topic, style, ...).
    """
    from_agent: str
    to_agent: str
    content: str
    metadata: dict = field(default_factory=dict)


def summarize_handoffs(handoffs):
    """Render the handoff chain as a human-readable text for debugging.

    Each line shows the sender, receiver, and the first 80 characters of the
    content, so you can see the full flow at a glance.
    """
    lines = []
    for h in handoffs:
        preview = h.content[:80].replace("\n", " ")
        lines.append(h.from_agent + " -> " + h.to_agent + ": " + preview)
    return "\n".join(lines)


## Task

`run_duo(task, researcher, writer) -> dict`

1. `findings = researcher.research(task)`
2. `h1 = Handoff('researcher', 'writer', findings, metadata={'task': task})`
3. `document = writer.write(findings)`
4. `h2 = Handoff('writer', 'user', document, metadata={'task': task})`
5. Return `{'findings': findings, 'document': document, 'handoffs': [h1, h2]}`

## Your Implementation

In [ ]:
def run_duo(task, researcher, writer):
    """Chain a ResearcherAgent then a WriterAgent for one task.
    Returns {'findings': str, 'document': str, 'handoffs': list[Handoff]}.
    """
    raise NotImplementedError


In [ ]:

# ── the researcher-writer pipeline ───────────────────────────────────────────
def run_duo(task, researcher, writer):
    """Chain a ResearcherAgent then a WriterAgent for one task.

    The researcher gathers findings; a Handoff carries them to the writer;
    the writer produces the document; a second Handoff records the output.

    Returns {"findings": str, "document": str, "handoffs": list[Handoff]}.
    """
    findings = researcher.research(task)
    h1 = Handoff("researcher", "writer", findings, metadata={"task": str(task)})
    document = writer.write(findings)
    h2 = Handoff("writer", "user", document, metadata={"task": str(task)})
    return {"findings": findings, "document": document, "handoffs": [h1, h2]}


## Automated checks

In [ ]:

score, total = 0, 5
try:
    researcher = ResearcherAgent(llm_fn=_mock_multi(findings='Key fact: X.'))
    writer = WriterAgent(llm_fn=_mock_multi(document='Final doc.'))

    result = run_duo('AI agents', researcher, writer)
    assert 'findings' in result and 'document' in result and 'handoffs' in result
    score += 1; print("✅ run_duo returns {findings, document, handoffs}")

    assert result['findings'] == 'Key fact: X.'
    assert result['document'] == 'Final doc.'
    score += 1; print("✅ findings and document are the agents' outputs")

    assert len(result['handoffs']) == 2
    score += 1; print("✅ two handoffs: researcher->writer and writer->user")

    h1, h2 = result['handoffs']
    assert h1.from_agent == 'researcher' and h1.to_agent == 'writer'
    assert h1.content == 'Key fact: X.'
    score += 1; print("✅ first handoff carries findings from researcher to writer")

    assert h2.from_agent == 'writer' and h2.to_agent == 'user'
    assert h2.content == 'Final doc.'
    score += 1; print("✅ second handoff carries document from writer to user")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python

# ── the researcher-writer pipeline ───────────────────────────────────────────
def run_duo(task, researcher, writer):
    """Chain a ResearcherAgent then a WriterAgent for one task.

    The researcher gathers findings; a Handoff carries them to the writer;
    the writer produces the document; a second Handoff records the output.

    Returns {"findings": str, "document": str, "handoffs": list[Handoff]}.
    """
    findings = researcher.research(task)
    h1 = Handoff("researcher", "writer", findings, metadata={"task": str(task)})
    document = writer.write(findings)
    h2 = Handoff("writer", "user", document, metadata={"task": str(task)})
    return {"findings": findings, "document": document, "handoffs": [h1, h2]}
```

**Why a free function and not a method?** `run_duo` takes agents as arguments rather than owning them — so you can pass any researcher and any writer, including mocks or future specialists. The Orchestrator (next exercise) *owns* the agents and calls `run_duo`; callers that want full control can skip the orchestrator and call `run_duo` directly.

</details>